In [122]:
!wget https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip
!unzip -q maestro-v3.0.0-midi.zip
!pip install pretty_midi

--2026-10-13 08:04:52--  https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 192.179.16.207, 173.194.64.207, 172.253.155.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|192.179.16.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 58416533 (56M) [application/octet-stream]
Saving to: ‘maestro-v3.0.0-midi.zip.2’

maestro-v3.0.0-midi 100%[===================>]  55.71M  89.8MB/s    in 0.6s    

2026-10-13 08:04:53 (89.8 MB/s) - ‘maestro-v3.0.0-midi.zip.2’ saved [58416533/58416533]

replace maestro-v3.0.0/2004/MIDI-Unprocessed_XP_08_R1_2004_01-02_ORIG_MID--AUDIO_08_R1_2004_01_Track01_wav.midi? [y]es, [n]o, [A]ll, [N]one, [r]ename: Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1369, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1340, in _find_and_load_unlocked
  File "<frozen importlib._bootstr

KeyboardInterrupt: 

In [62]:
import pretty_midi
from pathlib import Path

files = sorted(Path("maestro-v3.0.0").rglob("*.midi"))
print(len(files))   # should be around 1,276
print(files[0])

pm = pretty_midi.PrettyMIDI(str(files[0]))
piano = pm.instruments[0]

print(pm.get_end_time(), "seconds")
for note in piano.notes[:5]:
    print(note.pitch, note.start, note.end, note.velocity)

1276
maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi
969.0208333333334 seconds
71 1.0927083333333334 1.1895833333333332 60
55 1.2791666666666666 1.496875 44
59 1.4635416666666667 1.6312499999999999 55
62 1.6333333333333333 1.753125 52
71 1.2885416666666667 1.79375 54


In [74]:
import numpy as np
import torch

scale = 0.01
pieces = []

for file in files[:10]:
    pm = pretty_midi.PrettyMIDI(str(file))
    piano = next(inst for inst in pm.instruments if not inst.is_drum)

    n = int(round(pm.get_end_time() / scale)) + 1
    piece = np.zeros(n + 5000, dtype=np.float32)  # padding for nudged events


    for note in sorted(piano.notes, key=lambda x: x.start):
        start = int(round(note.start / scale))
        end = int(round(note.end / scale))

        while piece[start] != 0:
            start += 1
        end = max(end, start + 1)      # off event strictly after on event
        while piece[end] != 0:
            end += 1

        piece[start] = note.pitch + 120
        piece[end] = -note.pitch + 120

    pieces.append(torch.from_numpy(piece))

data = torch.cat(pieces).type(torch.long)

In [69]:
print((data != 0).sum().item() / len(data) * 100)

20.227097310455385


In [73]:
import numpy as np
import pretty_midi
from collections import defaultdict, deque
from IPython.display import Audio, display
import fluidsynth

def stream_to_midi(piece, scale=0.01, velocity=80, program=0):
    piece = np.asarray(piece)              # works for torch tensors on CPU too
    pm = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=program)   # 0 = acoustic grand piano

    open_notes = defaultdict(deque)        # pitch -> start times waiting for an off
    for idx in np.flatnonzero(piece):
        v = int(round(piece[idx]))
        t = idx * scale
        if v > 0:
            open_notes[v].append(t)
        else:
            pitch = -v
            if open_notes[pitch]:
                start = open_notes[pitch].popleft()   # FIFO pairing
                inst.notes.append(pretty_midi.Note(velocity, pitch, start, t))

    pm.instruments.append(inst)
    return pm

pm_out = stream_to_midi(data[1].numpy() if hasattr(piece, "numpy") else piece, scale=0.01)

# Listen in the notebook
sf2 = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
audio = pm_out.fluidsynth(fs=44100, sf2_path=sf2)
display(Audio(audio, rate=44100))

/tmp/ipykernel_24606/3489307696.py:31: DeprecationWarning: The parameter 'sf2_path' is deprecated, please use 'synthesizer' instead.
  audio = pm_out.fluidsynth(fs=44100, sf2_path=sf2)


ValueError: zero-size array to reduction operation maximum which has no identity

In [ ]:
#ORIGINAL AUDIO
audio = pm.fluidsynth(fs=44100, sf2_path=sf2)
display(Audio(audio, rate=44100))

In [ ]:
n = int(0.9*len(data))
data_train = data[:n]
data_test = data[n:]

In [ ]:
#manual seef
torch.manual_seed(1502)

In [ ]:
BATCH_SIZE = 4
BLOCK_SIZE = 50

def get_batch(data, batch_size, block_size):
    ix = torch.randint(len(data)-block_size, (batch_size, ))
    print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch(data_train, BATCH_SIZE, BLOCK_SIZE)

In [66]:
vocab_size = len(data_train.unique())

In [67]:
print(xb)

tensor([[  0,   0,   0,   0, -72,   0,  63,  60,   0,   0,   0,   0,   0,   0,
         -63, -60,  68,  51,   0,   0,   0,   0, -68,  72,   0, -51,   0,   0,
           0,   0,   0,   0, -72,  44,  75,   0,   0,   0, -44,   0,   0,   0,
          51,   0,   0,  68,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,  83,   0,   0,   0,   0,   0,   0, -83,   0,   0,
           0,  92,   0,   0,   0,  83,   0,   0,   0,   0,   0,   0,   0, -83,
          56,  88, -92, -52,   0,   0,   0,   0, -88,   0,   0,   0,  80,   0,
           0,   0,   0, -80,   0,   0,   0,   0],
        [ 57,  79, -74, -78, -57,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,  78, -79,   0,   0,   0,   0,   0,  

In [59]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLangModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]               # (B, C)
            probs = F.softmax(logits, dim=-1)       # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLangModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)   # (B*T, vocab_size)
print(loss)

IndexError: index out of range in self

In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch(data_train, BATCH_SIZE, BLOCK_SIZE)

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())
